# Module 2 - Foundation models (programmatic AI Playground)

The AI Playground (Workspace > AI Playground) is the click-through way to send prompts to foundation models. This notebook is the **same thing without the UI** so the entire workshop runs cell-by-cell.

We will:
1. List foundation model endpoints available in this workspace
2. Call `ai_query` from SQL
3. Call the chat-completions API directly via the Python SDK
4. Compare Claude Haiku 4.5 (fast, cheap) vs. Claude Sonnet 4.6 (smarter)

In [ ]:
%run ./_config


## 1. List foundation model endpoints

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
fm_endpoints = []
for ep in w.serving_endpoints.list():
    name = ep.name or ""
    if name.startswith("databricks-") and any(t in name for t in ("claude", "llama", "gemma", "mistral", "gte", "bge")):
        fm_endpoints.append(name)
for n in sorted(fm_endpoints):
    print(" ", n)

## 2. ai_query from SQL

`ai_query` is the SQL-side wrapper. Use it directly inside Spark SQL or DBSQL queries to call any model serving endpoint.

In [ ]:
%sql
SELECT ai_query(
  'databricks-gpt-5-mini',
  'In two sentences, what is the difference between a CVE and an entry in the CISA KEV catalog?'
) AS answer

## 3. Chat completions via the SDK

Direct access to the chat-completions API. Useful when you want full control over messages, tool calls, temperature, etc.

In [ ]:
# Direct call to model serving via SDK api_client.do()
# (works on the SDK shipped with DBR; does NOT require databricks-sdk[openai] extras)
def chat(model, messages, max_tokens=400, tools=None):
    body = {"messages": messages, "max_tokens": max_tokens}
    if tools is not None:
        body["tools"] = tools
    return w.api_client.do("POST", f"/serving-endpoints/{model}/invocations", body=body)

msg = [
    {"role": "system", "content": "You are a senior cyber threat intel analyst. Be concise."},
    {"role": "user", "content": "Explain ProxyShell-class Exchange Server vulnerabilities in three bullets."},
]
resp = chat("databricks-gpt-5-2", msg)
print(resp["choices"][0]["message"]["content"])


## 4. Side-by-side: Haiku 4.5 vs Sonnet 4.6

Same prompt, two models. Notice the latency and depth difference. Haiku is the right pick for high-volume parsing (notebook 1's `ai_parse_document`). Sonnet is the right pick for the supervisor agent (notebook 5).

In [ ]:
import time
PROMPT = "A KEV entry was just added for CVE-2024-21762 affecting FortiOS SSL-VPN. In 4 bullets, what should a DISA SOC do in the next hour?"

for model in ["databricks-gpt-5-mini", "databricks-gpt-5-2"]:
    t0 = time.monotonic()
    out = chat(model, [{"role": "user", "content": PROMPT}], max_tokens=300)
    dt = time.monotonic() - t0
    print("=" * 60)
    print(f"{model}  ({dt:.1f}s)")
    print("=" * 60)
    print(out["choices"][0]["message"]["content"])
    print()


## 5. Tool use (preview)

The same chat API supports tool calls. Notebook 5 builds a full tool-using supervisor agent; this is the minimum viable example.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "lookup_cvss",
        "description": "Return the CVSS score for a CVE id.",
        "parameters": {
            "type": "object",
            "properties": {"cve_id": {"type": "string"}},
            "required": ["cve_id"],
        },
    },
}]
out = chat(
    "databricks-gpt-5-2",
    [{"role": "user", "content": "What is the CVSS score for CVE-2024-21762?"}],
    max_tokens=200,
    tools=tools,
)
msg = out["choices"][0]["message"]
print("content   :", msg.get("content"))
print("tool calls:", msg.get("tool_calls"))
